# 12 — Bayesian Nonparametrics
**References:** Hjort et al. (2010) *Bayesian Nonparametrics* Ch. 1–3 · Teh (2010) · Ferguson (1973)

## Narrative thread
```
Motivation -> Dirichlet process -> CRP -> DP mixtures -> Stick-breaking -> Posterior inference
```

## 1. Motivation: models that grow with data

In parametric models, the number of parameters is fixed before seeing data. **Bayesian nonparametrics (BNP)** uses models with **infinitely many parameters** — the effective complexity grows with $n$.

**Key BNP objects:**

| Object | Used as prior for | Key property |
|---|---|---|
| Dirichlet process $\text{DP}(\alpha, H)$ | Distribution over distributions | Discrete with probability 1 |
| Chinese Restaurant Process (CRP) | Partition / clustering | Predictive view of DP |
| Pitman-Yor process | Language models, power-law | Generalizes DP |
| Beta process | Feature allocation | Indian Buffet Process (IBP) |
| Gaussian process | Distribution over functions | Covered in notebook 11 |

## 2. The Dirichlet process

$G \sim \text{DP}(\alpha, H)$ where $\alpha > 0$ (concentration) and $H$ (base measure) satisfies:

For any partition $\{A_1,\ldots,A_k\}$ of the sample space:
$$(G(A_1), \ldots, G(A_k)) \sim \text{Dirichlet}(\alpha H(A_1), \ldots, \alpha H(A_k))$$

**Key properties:**
- $E[G(A)] = H(A)$ — $H$ is the prior guess for $G$
- $\text{Var}[G(A)] = H(A)(1-H(A))/(\alpha+1)$ — large $\alpha$ → concentrated near $H$
- $G$ is **discrete** a.s. — draws from $G$ repeat (clustering property)

**Stick-breaking representation** (Sethuraman 1994):
$$G = \sum_{k=1}^\infty w_k\, \delta_{\phi_k}, \quad w_k = v_k \prod_{j<k}(1-v_j), \quad v_k \overset{\text{iid}}{\sim} \text{Beta}(1, \alpha), \quad \phi_k \overset{\text{iid}}{\sim} H$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

def stick_breaking(alpha, K=100):
    """Sample weights from DP(alpha, H) via stick-breaking (truncated at K)."""
    v = np.random.beta(1, alpha, K)
    v[-1] = 1.0  # ensure weights sum to 1
    w = v * np.concatenate([[1], np.cumprod(1 - v[:-1])])
    return w

# ── Stick-breaking: effect of alpha ──────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
alphas = [0.5, 2, 10]
K = 30
x_base = np.linspace(-5, 5, 400)

for col, alpha in enumerate(alphas):
    # Draw 5 realizations of G ~ DP(alpha, N(0,1))
    ax0 = axes[0, col]
    for _ in range(5):
        w      = stick_breaking(alpha, K)
        phi    = np.random.normal(0, 1, K)  # atoms from H = N(0,1)
        idx    = np.argsort(phi)
        ax0.step(phi[idx], np.cumsum(w[idx]), where='post', lw=1.5, alpha=0.7)
    ax0.plot(x_base, stats.norm.cdf(x_base), 'k--', lw=2, label='Base measure H = N(0,1)')
    ax0.set_title(f'DP(α={alpha}): realizations of G\n{"Few" if alpha<2 else "Many"} large atoms = {"clustered" if alpha<2 else "spread"}')
    ax0.set_xlabel('x'); ax0.set_ylabel('CDF'); ax0.set_xlim(-4, 4)
    ax0.legend(fontsize=7)

    # Weight distribution
    ax1 = axes[1, col]
    all_weights = np.array([stick_breaking(alpha, K) for _ in range(200)])
    eff_k = np.array([np.sum(w > 0.01) for w in all_weights])
    ax1.hist(eff_k, bins=range(1, K+1), density=True, color='#4361ee', alpha=0.8, edgecolor='white')
    ax1.set_xlabel('Effective # clusters (w > 0.01)')
    ax1.set_ylabel('Frequency')
    ax1.set_title(f'α={alpha}: distribution of effective cluster count\nE[clusters] ≈ {eff_k.mean():.1f}')

plt.suptitle('Dirichlet Process: Stick-Breaking Representation\n'
             'Large α → more clusters; small α → few dominant clusters',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Chinese Restaurant Process

The **CRP** gives the predictive distribution for the partition induced by a DP.

Given $n-1$ customers seated at tables, customer $n$ sits at:
- **Existing table $k$** with probability $\propto n_k$ (number already there)
- **New table** with probability $\propto \alpha$

$$P(z_n = k \mid z_1,\ldots,z_{n-1}) = \begin{cases} \frac{n_k}{n - 1 + \alpha} & k \text{ existing} \\ \frac{\alpha}{n - 1 + \alpha} & k \text{ new} \end{cases}$$

**Rich get richer:** Popular tables attract more customers — heavy-tailed cluster size distribution.

**Expected number of clusters:** $E[K_n] \approx \alpha \log(n/\alpha + 1)$ — grows logarithmically with $n$.

In [ ]:
# ── Chinese Restaurant Process simulation ─────────────────────────────────
def crp(n, alpha):
    """Simulate n customers under CRP(alpha). Returns cluster assignments."""
    assignments = [0]  # first customer at table 0
    table_counts = [1]
    for i in range(1, n):
        probs = np.array(table_counts + [alpha])
        probs = probs / probs.sum()
        z = np.random.choice(len(probs), p=probs)
        if z == len(table_counts):  # new table
            table_counts.append(1)
        else:
            table_counts[z] += 1
        assignments.append(z)
    return np.array(assignments), np.array(table_counts)

n_customers = 200
alphas_crp  = [0.5, 2, 10]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))

for col, alpha in enumerate(alphas_crp):
    z, counts = crp(n_customers, alpha)
    K_final   = len(counts)

    # Cluster sizes
    axes[0, col].bar(range(K_final), sorted(counts, reverse=True),
                     color='#4361ee', alpha=0.8)
    axes[0, col].set_xlabel('Cluster (sorted by size)')
    axes[0, col].set_ylabel('# customers')
    axes[0, col].set_title(f'CRP(α={alpha}): {K_final} clusters\nn={n_customers} customers')

    # Growth of # clusters over time
    k_over_time = [len(set(z[:t+1])) for t in range(n_customers)]
    axes[1, col].plot(range(n_customers), k_over_time, color='#4361ee', lw=2)
    ns = np.arange(1, n_customers+1)
    axes[1, col].plot(ns, alpha * np.log(ns/alpha + 1), 'r--', lw=2, label=f'$\\alpha\\log(n/\\alpha+1)$')
    axes[1, col].set_xlabel('n (customers)'); axes[1, col].set_ylabel('# clusters')
    axes[1, col].set_title(f'Cluster growth: α={alpha}\n# clusters ~ α log(n)')
    axes[1, col].legend(fontsize=8)

plt.suptitle('Chinese Restaurant Process: Rich-get-richer Clustering\n'
             '# clusters grows as O(α log n) — unbounded but slow',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. DP Mixture Models

The **Dirichlet Process Mixture Model** (DPMM) is a Bayesian infinite mixture:

$$G \mid \alpha, H \sim \text{DP}(\alpha, H)$$
$$\theta_i \mid G \sim G$$
$$x_i \mid \theta_i \sim F(\theta_i)$$

Because $G$ is discrete, multiple $\theta_i$ are identical, inducing a clustering structure. The number of clusters is inferred from the data.

**Collapsed Gibbs sampler** uses the CRP predictive:
$$p(z_i = k \mid \mathbf{z}_{-i}, \mathbf{x}) \propto \begin{cases} n_{-i,k} \cdot p(x_i \mid \mathbf{x}_{-i,k}) & \text{existing} \\ \alpha \cdot p(x_i) & \text{new} \end{cases}$$

## 5. Key takeaways

| Concept | Statement |
|---|---|
| DP | Prior over distributions; discrete a.s. |
| $\alpha$ | Concentration: large $\alpha$ → many small clusters |
| Stick-breaking | Constructive representation: infinite weighted atoms |
| CRP | Predictive view: rich-get-richer clustering |
| DPMM | Infinite mixture — number of clusters inferred from data |
| BNP advantage | Model complexity adapts to data; no need to pre-specify $K$ |

> **End of course.** Topics covered: foundations, conjugate models, priors, posterior computation, MH MCMC, Gibbs/HMC, model checking, hierarchical models, regression with shrinkage priors, variational inference, Gaussian processes, and Bayesian nonparametrics.